# 02 — Feature engineering

Apply the wearable / smartphone / climate / missingness / baseline modules
and inspect what we get.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from lhfm.data.synthetic_generator import generate_synthetic_cohort
from lhfm.features import build_full_feature_table

In [ ]:
raw = generate_synthetic_cohort(n_participants=80, n_days=60, seed=11)
feat = build_full_feature_table(raw, impute=True, add_targets=True)
print('raw     :', raw.shape)
print('featured:', feat.shape)
new_cols = sorted(set(feat.columns) - set(raw.columns))
print(f'{len(new_cols)} new columns')
new_cols

### Target prevalences
Realistic class imbalance is critical. We want these in the 5-15% range.

In [ ]:
tcols = [c for c in feat.columns if c.startswith('target_')]
feat[tcols].mean().to_frame('positive_rate')

### Within-person z-scores: sanity check

In [ ]:
# Within-person, screen_time_z should average to near zero.
feat.groupby('participant_id')['screen_time_z'].mean().describe()

### Missingness features in action

In [ ]:
pid = feat['participant_id'].iloc[0]
sub = feat[feat.participant_id == pid].sort_values('date')
sub['date'] = pd.to_datetime(sub['date'])
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(sub.date, sub.any_missing);             axes[0].set_ylabel('any missing')
axes[1].plot(sub.date, sub.consecutive_missing_days);axes[1].set_ylabel('streak')
axes[2].plot(sub.date, sub.missingness_rate_7d);     axes[2].set_ylabel('7d rate')
for ax in axes: ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

### Recovery score vs survey mood

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
mask = feat.survey_mood.notna()
ax.scatter(feat.loc[mask,'recovery_score'], feat.loc[mask,'survey_mood'], alpha=0.15, s=8)
ax.set_xlabel('recovery_score'); ax.set_ylabel('survey_mood')
ax.grid(alpha=0.3); plt.show()
corr = feat.loc[mask, ['recovery_score','survey_mood']].corr().iloc[0,1]
print(f'correlation = {corr:.3f}')